In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA version (built with):", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.12.0.dev20260317+cu128
CUDA version (built with): 12.8
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Ti


In [2]:
import torch
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

x = torch.randn(2000, 2000, device="cuda")
y = torch.matmul(x, x)

print("Success:", y.shape)

Success: torch.Size([2000, 2000])


In [3]:
import torch
print(torch.version.cuda)
print(torch.cuda.get_arch_list())


12.8
['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']


In [4]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

a = torch.randn(4000, 4000, device="cuda")
b = torch.randn(4000, 4000, device="cuda")
c = torch.matmul(a, b)

print("Done:", c.shape)
print("Memory allocated (GB):", torch.cuda.memory_allocated() / 1e9)

CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Ti
VRAM (GB): 17.102864384
Done: torch.Size([4000, 4000])
Memory allocated (GB): 0.262144


In [5]:
import torch
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(1024, 2048),
    nn.ReLU(),
    nn.Linear(2048, 10)
).cuda()

x = torch.randn(256, 1024, device="cuda")
y = torch.randint(0, 10, (256,), device="cuda")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for i in range(5):
    optimizer.zero_grad()
    out = model(x)
    loss = criterion(out, y)
    loss.backward()
    optimizer.step()
    print(i, loss.item())

0 2.3206028938293457
1 1.4456021785736084
2 0.8065009117126465
3 0.39490365982055664
4 0.17626464366912842


In [6]:
import torch
import time

size = 4000

# CPU
x_cpu = torch.randn(size, size)
start = time.time()
y_cpu = torch.matmul(x_cpu, x_cpu)
cpu_time = time.time() - start

# GPU
x_gpu = torch.randn(size, size, device="cuda")
torch.cuda.synchronize()

start = time.time()
y_gpu = torch.matmul(x_gpu, x_gpu)
torch.cuda.synchronize()
gpu_time = time.time() - start

print(f"CPU time: {cpu_time:.4f} sec")
print(f"GPU time: {gpu_time:.4f} sec")
print(f"Speedup: {cpu_time/gpu_time:.2f}x")

CPU time: 0.1290 sec
GPU time: 0.0550 sec
Speedup: 2.35x


In [7]:
import torch
import torch.nn as nn
import time

device_gpu = "cuda"
device_cpu = "cpu"

model_gpu = nn.Sequential(
    nn.Linear(1024, 2048),
    nn.ReLU(),
    nn.Linear(2048, 10)
).to(device_gpu)

model_cpu = nn.Sequential(
    nn.Linear(1024, 2048),
    nn.ReLU(),
    nn.Linear(2048, 10)
).to(device_cpu)

x_gpu = torch.randn(512, 1024, device=device_gpu)
y_gpu = torch.randint(0, 10, (512,), device=device_gpu)

x_cpu = x_gpu.cpu()
y_cpu = y_gpu.cpu()

criterion = nn.CrossEntropyLoss()

# GPU timing
torch.cuda.synchronize()
start = time.time()

out = model_gpu(x_gpu)
loss = criterion(out, y_gpu)
loss.backward()

torch.cuda.synchronize()
gpu_time = time.time() - start

# CPU timing
start = time.time()

out = model_cpu(x_cpu)
loss = criterion(out, y_cpu)
loss.backward()

cpu_time = time.time() - start

print(f"CPU time: {cpu_time:.4f} sec")
print(f"GPU time: {gpu_time:.4f} sec")
print(f"Speedup: {cpu_time/gpu_time:.2f}x")

CPU time: 0.0100 sec
GPU time: 0.0050 sec
Speedup: 2.00x


In [10]:
import torch, torch.nn as nn, time

device = "cuda"

model = nn.Sequential(
    nn.Linear(4096, 8192),
    nn.ReLU(),
    nn.Linear(8192, 4096),
).to(device)

x = torch.randn(4096, 4096, device=device)
y = torch.randn(4096, 4096, device=device)

start = time.time()

for _ in range(10):
    out = model(x)
    loss = (out - y).pow(2).mean()
    loss.backward()

torch.cuda.synchronize()
gpu_time = time.time() - start

print("GPU training time:", gpu_time)

GPU training time: 1.1865074634552002
